In [ ]:
R = 200
r = 2  # geometric radius in coordinate units
p = 50
h = 80    # vertical scale

# Define a genuinely non-planar 5-gon in 3D (intersection points)
z_offsets = [0, 1, -0.5, 1.2, -1.5]  # chosen so points are not coplanar
vertices = [
    (R*cos(2*pi*k/5), R*sin(2*pi*k/5), h*z_offsets[k])
    for k in range(5)
]

# Build extended "full" lines through each pair of adjacent vertices
extend_factor = 2  # how far past each vertex to extend
segments = []
for k in range(5):
    v1 = vector(vertices[k])
    v2 = vector(vertices[(k+1) % 5])
    d = v2 - v1
    p_ext = tuple(v1 - extend_factor * d)
    q_ext = tuple(v2 + extend_factor * d)
    segments.append((p_ext, q_ext))

G = sum((LineSegment(p, q, radius=r) for p, q in segments), 0)

# Spheres only at the intersection points (vertices)
G += sum((sphere(v, r) for v in vertices), 0)

show(G + P_plot, frame=False)

In [ ]:
from sage.all import *
from sage.plot.plot3d.shapes import LineSegment

R = 200
r = 2   # geometric radius in coordinate units
p = 50
h = 80  # vertical scale

# 5 vertices → 4-line chain
z_offsets = [0, 1, -0.5, 1.2, -1.5]  # chosen so points are not coplanar
vertices = [
    (R*cos(2*pi*k/5), R*sin(2*pi*k/5), h*z_offsets[k])
    for k in range(5)
]

# Build extended "full" lines through consecutive vertex pairs
extend_factor = 2  # how far past each vertex to extend
segments = []
for k in range(4):           # 4 lines: l1,l2,l3,l4
    v1 = vector(vertices[k])
    v2 = vector(vertices[k+1])
    d = v2 - v1
    p_ext = tuple(v1 - extend_factor * d)
    q_ext = tuple(v2 + extend_factor * d)
    segments.append((p_ext, q_ext))

# Lines
G_lines = sum(LineSegment(p, q, radius=r, color='black')
              for p, q in segments)

# Spheres at intersection points (inner vertices v1, v2, v3)
G_spheres = sum(sphere(vertices[k], r, color='red') for k in range(1, 4))

# Planes H1 (l1,l2) and H2 (l3,l4) as finite patches
u, v = var('u v')
plane_extent = 1200

def plane_patch(segA, segB, color):
    p1, q1 = map(vector, segA)
    p2, q2 = map(vector, segB)
    d1 = (q1 - p1).normalized()
    d2 = (q2 - p2).normalized()
    base = (p1 + p2) / 2   # point in the plane

    fx = lambda u, v: base[0] + u*d1[0] + v*d2[0]
    fy = lambda u, v: base[1] + u*d1[1] + v*d2[1]
    fz = lambda u, v: base[2] + u*d1[2] + v*d2[2]

    return parametric_plot3d(
        (fx, fy, fz),
        (u, -plane_extent, plane_extent),
        (v, -plane_extent, plane_extent),
        color=color, opacity=0.3, mesh=False
    )

l1, l2, l3, l4 = segments
H1 = plane_patch(l1, l2, color='blue')
H2 = plane_patch(l3, l4, color='green')

show(G_lines + G_spheres + H1 + H2, frame=False)